<center><ins><h1>Sinking Properties</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Sinkings properties are a combination of sinking velocity, particle size and recovery rate plotted as a radar plot.</li>
    <li>Various units (%, µm, m/h)</li>
    <li>Files are imported over .csv from raw data analysis in the respective notebooks.</li>
</ul>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

#### (Optional: TODO By Time not yet working with correct subtexts + manual color_palette change)

# Imports

In [1]:
import os, sys

WORKSPACE = os.path.abspath(os.path.join(os.getcwd(), ".."))
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

from scripts import data_helper, plot_helper
import pandas as pd

CONFIG = data_helper.load_config()
META_DATA = data_helper.load_meta_data(CONFIG)
plot_helper.set_config(CONFIG)

# Variables

In [2]:

IGNORE_EPS = True

DISPLAY_TITLE = False
ALIGNMENT = "Vertical" # radar plot alignment either Horizontal | Vertical
TOGGLE_TYPE = "SAMPLE_NAME" 

PLOT_FACECOLOR = "w" # default w
PLOT_FONTCOLOR = "k" # default k
PLOT_AXESCOLOR = "k" # default k

# Show numeric output in decimal format e.g., 2.1514
pd.options.display.float_format = '{:,.2f}'.format

# Raw Import

In [3]:
# Get all raw data files
rec_rat = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Recovery-Rates_Raw-Data.csv"))
sin_vel = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Sinking-Velocities_Raw-Data.csv"))
par_siz = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Particle-Sizes_Raw-Data.csv"))

# Merge into dataframe with all categories
merged_df = pd.concat([rec_rat["EXPERIMENT_NAME"], rec_rat["SAMPLE_NAME"], rec_rat["TIME"], rec_rat["RRATE_%"], sin_vel["MEAN_SVELO_M/H"], par_siz["GEOMETRIC.MEAN"]], axis = 1)
# Calculate mean of triplicates
mean_df = merged_df.groupby(["EXPERIMENT_NAME", "SAMPLE_NAME", "TIME"], as_index = False).agg({"RRATE_%": pd.Series.mean, "MEAN_SVELO_M/H": pd.Series.mean, "GEOMETRIC.MEAN": pd.Series.mean})

# Delete +EPS data
if IGNORE_EPS:
    mean_df.drop(mean_df[mean_df["SAMPLE_NAME"] == "+EPS"].index, inplace = True)

mean_df.head()

,EXPERIMENT_NAME,SAMPLE_NAME,TIME,RRATE_%,MEAN_SVELO_M/H,GEOMETRIC.MEAN
0,SC,A.obliquus,0.00,1.81,0.00,780.86
1,SC,Algaemix,0.00,97.48,4.16,780.86
2,SC,C.reinhardtii,0.00,4.60,0.52,780.86
3,SC,E.texensis,0.00,18.57,1.03,780.86
4,SC,T.albertanoae,0.00,26.07,0.71,780.86


# Plot Visualization 

In [4]:
# Filter unneeded time points
plot_df = pd.DataFrame(data_helper.filter(mean_df, time = [12]))

# Iterate through all experiments and create radar plots
for exp in data_helper.get_experiment_types(META_DATA, CONFIG):
    print(f"Working on: {exp}")
    # Grab subets of data and meta inf
    sub_exp_df = plot_df[plot_df["EXPERIMENT_NAME"] == exp]
    sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
    #if IGNORE_EPS & (exp == "RM") : # Delete +EPS sample for graphs
    #    sub_meta["SAMPLES_APPENDIX"] = ','.join([sample_appendix for index, sample_appendix in enumerate(sub_meta["SAMPLES_APPENDIX"][0].split(",")) if index != 2])
    # Custom sample sorting
    sub_exp_df["SAMPLE_NAME"] = pd.Categorical(sub_exp_df["SAMPLE_NAME"], categories = sub_meta["SAMPLES_ORDER"][0].split(","))
    sub_exp_df = sub_exp_df.sort_values(by = "SAMPLE_NAME")
    
    plot_helper.visualize_radarplotseries(sub_exp_df, sub_meta,
                                          DISPLAY_TITLE, 
                                          TOGGLE_TYPE, ALIGNMENT,
                                          PLOT_AXESCOLOR, PLOT_FACECOLOR, PLOT_FONTCOLOR)

Working on: SC
Creating single radar plot for: A.obliquus
Creating single radar plot for: C.reinhardtii
Creating single radar plot for: E.texensis
Creating single radar plot for: T.albertanoae
Creating single radar plot for: Algaemix
Radar-Series "SC" created and saved.


# ARCHIVE

[Source Code for Radar Plot Class](url=https://stackoverflow.com/questions/19826142/it-is-possible-to-round-up-integers-using-format)